1/x in encoder1 25030.75, 289 degrees
1/x in encoder2 6514.89, 228 degrees

In [1]:
import subprocess
from datasets import load_dataset
import re
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

dataset = load_dataset("dair-ai/emotion", split="test")

/Users/tonyma/code/FHE-BERT-Tiny-Emotion/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gokuls/BERT-tiny-emotion-intent")

texts = dataset['text']
token_length = [len(tokenizer.encode(entry['text'], truncation=False)) for entry in dataset]
labels = dataset['label']

df = pd.DataFrame({
    'text': texts,
    'token_length': token_length,
    'label': labels
})

# sort label
df_label = df.sort_values(by=['label', 'token_length'], ascending=[True, True]).reset_index(drop=True)

In [3]:
new_df2 = df_label.groupby('label').head(10).reset_index(drop=True)
print(len(new_df2))
print(new_df2[:21])

60
                            text  token_length  label
0          i feel so embarrassed             6      0
1             i do feel stressed             6      0
2                i feels so lame             6      0
3      i feel embarrassed enough             6      0
4          i stop feeling guilty             6      0
5         i feel depressed again             6      0
6              i feel soo lonely             6      0
7     im feeling depressed again             6      0
8           i just feel troubled             6      0
9              i feel less alone             6      0
10          im feeling energetic             5      1
11          i feel more creative             6      1
12        i feel really inspired             6      1
13         i feel more energetic             6      1
14             i feel any better             6      1
15       i would feel productive             6      1
16   im feeling hopeful relieved             6      1
17           i feel gorge

In [4]:
correct = []
error = []
output = []
execution_times = []

func_error = []
count = 0

def run_fhe_bert_tiny(sample):
    global correct, error, func_error, output, execution_times, count
    #print(sample)
    
    sentence = sample.text
    print("sentence: ", sentence)
    
    cmd = [f"./FHE-BERT-Tiny", sentence]
    #cmd = ["./FHE-BERT-Tiny", sentence, "--verbose"]
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )
    
    # format
    result = result.stdout.strip()
    #print("raw result", result)
    
    # check [x, x, x, x, x, x]
    #pattern = r'\[(.*?)\]'
    pattern = r'plain_result: \[(.*?)\]'
    match = re.search(pattern, result)
    tmp = match.group(1).split(",")
    res = [float(x) for x in tmp]
    output.append(res)
    print(res)
    
    # execution time
    # The evaluation of the FHE circuit took: 121 seconds.
    pattern_time = r'The evaluation of the FHE circuit took:\s*(\d+)\s*seconds'
    match_time = re.search(pattern_time, result)
    seconds = float(match_time.group(1))
    execution_times.append(seconds)
        
    if ("Outcome: sadness" in result):
        if (sample.label == 0):
            correct.append(count)
            print("sadness")
        else:
            error.append(count)
    elif ("Outcome: joy" in result):
        if (sample.label == 1):
            correct.append(count)
            print("joy")
        else:
            error.append(count)
    elif ("Outcome: love" in result):
        if (sample.label == 2):
            correct.append(count)
            print("love")
        else:
            error.append(count)
    elif ("Outcome: anger" in result):
        if (sample.label == 3):
            correct.append(count)
            print("anger")
        else:
            error.append(count)
    elif ("Outcome: fear" in result):
        if (sample.label == 4):
            correct.append(count)
            print("fear")
        else:
            error.append(count)
    elif ("Outcome: surprise" in result):
        if (sample.label == 5):
            correct.append(count)
            print("surprise")
        else:
            error.append(count)
    else: # error
        func_error.append(count)
        
    print(seconds, "seconds")

In [5]:
from tqdm import tqdm

for row in tqdm(new_df2.itertuples(index=False), total=new_df2.shape[0]):
    run_fhe_bert_tiny(row)
    count+=1

  0%|          | 0/60 [00:00<?, ?it/s]

sentence:  i feel so embarrassed


  2%|▏         | 1/60 [02:31<2:28:30, 151.03s/it]

[5.69863e+49, -2.06124e+49, -1.20339e+49, 5.01869e+49, 6.29484e+49, 5.72924e+49]
139.0 seconds
sentence:  i do feel stressed


  3%|▎         | 2/60 [05:00<2:25:18, 150.32s/it]

...

[1.34187e+49, 3.16199e+49, 4.02043e+49, 3.07775e+49, -8.70392e+48, -9.79071e+49]
189.0 seconds
sentence:  i am feeling amazing and seeing the difference


 98%|█████████▊| 59/60 [2:40:00<03:10, 190.79s/it]

[-3.92108e+49, -2.54222e+49, -8.1745e+48, -1.29994e+49, 4.6498e+49, -2.9966e+48]
184.0 seconds
sentence:  i always feel very shocked by that me threatening


100%|██████████| 60/60 [2:43:24<00:00, 163.41s/it]

[2.38484e+49, 2.96364e+49, 9.38313e+49, 2.13301e+49, 2.95986e+49, -8.4063e+47]
193.0 seconds


In [6]:
len(correct)

9

In [7]:
len(error)

51

In [8]:
error

[0,
 1,
 2,
 3,
 4,
 5,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 26,
 27,
 28,
 30,
 31,
 32,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 44,
 45,
 46,
 47,
 48,
 49,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59]

In [9]:
print(len(output))
print(len(execution_times))

60
60


In [10]:
import numpy as np
np.savetxt("output_labels4_60", output, delimiter=",")
np.savetxt("execution_time_labels4_60", output, delimiter=",")

Test again

In [11]:
import math

new_resamples = []
for i in range(len(output)):
    if i in error:
        if (int(math.floor(math.log10(abs(output[i][0])))) > 0) or (int(math.floor(math.log10(abs(output[i][1])))) > 0) or (int(math.floor(math.log10(abs(output[i][2])))) > 0) or (int(math.floor(math.log10(abs(output[i][3])))) > 0) or (int(math.floor(math.log10(abs(output[i][4])))) > 0) or (int(math.floor(math.log10(abs(output[i][5])))) > 0):
            print(f"{i}: {output[i]}")
            new_resamples.append(i)
            
print(len(new_resamples))

0: [5.69863e+49, -2.06124e+49, -1.20339e+49, 5.01869e+49, 6.29484e+49, 5.72924e+49]
1: [3.82212e+49, 5.53353e+49, -4.22711e+48, 1.88815e+49, -1.33213e+49, -2.46055e+49]
2: [-1.16584e+48, -5.1292e+49, 3.81851e+49, -3.54781e+49, 4.13336e+49, -3.14962e+48]
3: [7.90639e+48, 4.62542e+49, -8.74293e+48, -2.92132e+49, 3.07521e+49, 3.18034e+49]
4: [1.94879e+49, 2.86678e+49, -4.62297e+49, -3.37907e+49, -1.81598e+49, 2.46072e+49]
5: [1.39257e+49, -7.47774e+49, 2.76157e+49, -4.10241e+49, 3.2083e+49, -2.511e+49]
8: [5.37873e+49, 4.11065e+49, 3.86843e+49, 7.64094e+49, -2.06102e+49, 2.26491e+49]
9: [-3.06167e+49, -1.83223e+49, 1.52407e+49, 3.45298e+49, -6.182e+49, 7.40682e+49]
10: [-4.69023e+49, -2.6516e+49, 3.55536e+49, 1.31446e+49, 5.27938e+49, -2.55328e+49]
11: [4.01503e+49, -2.16299e+49, -4.96493e+49, 6.27312e+49, 2.65446e+49, 5.48983e+48]
12: [-3.11249e+49, 1.2372e+49, -7.96035e+49, 3.10377e+49, -7.33164e+49, 1.80007e+48]
13: [-4.60285e+49, -1.7692e+49, 6.42723e+49, 1.06253e+50, 1.16142e+50, 2.9

In [12]:
new_resamples

[0,
 1,
 2,
 3,
 4,
 5,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 26,
 27,
 28,
 30,
 31,
 32,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 44,
 45,
 46,
 47,
 48,
 49,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59]

In [13]:
# check again
for i in new_resamples:
    if not i in error:
        print("not exist: ", i)
        #new_resamples.remove(i)

In [14]:
correct2 = []
error2 = []
output2 = []
execution_times2 = []

func_error2 = []
count2 = 0

def run_fhe_bert_tiny2(sample):
    global correct2, error2, func_error2, output2, execution_times2, count2
    #print(sample)
    
    sentence = sample.text
    print("sentence: ", sentence)
    
    #cmd = [f"./FHE-BERT-Tiny", sentence]
    cmd = ["./FHE-BERT-Tiny", sentence, "--verbose"]
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )
    
    # format
    result = result.stdout.strip()
    #print("raw result", result)
    
    # check [x, x, x, x, x, x]
    #pattern = r'\[(.*?)\]'
    pattern = r'plain_result: \[(.*?)\]'
    match = re.search(pattern, result)
    tmp = match.group(1).split(",")
    res = [float(x) for x in tmp]
    output2.append(res)
    print(res)
    
    # execution time
    # The evaluation of the FHE circuit took: 121 seconds.
    pattern_time = r'The evaluation of the FHE circuit took:\s*(\d+)\s*seconds'
    match_time = re.search(pattern_time, result)
    seconds = float(match_time.group(1))
    execution_times2.append(seconds)
        
    if ("Outcome: sadness" in result):
        if (sample.label == 0):
            correct2.append(count2)
            print("sadness")
        else:
            error2.append(count2)
    elif ("Outcome: joy" in result):
        if (sample.label == 1):
            correct2.append(count2)
            print("joy")
        else:
            error2.append(count2)
    elif ("Outcome: love" in result):
        if (sample.label == 2):
            correct2.append(count2)
            print("love")
        else:
            error2.append(count)
    elif ("Outcome: anger" in result):
        if (sample.label == 3):
            correct2.append(count2)
            print("anger")
        else:
            error2.append(count)
    elif ("Outcome: fear" in result):
        if (sample.label == 4):
            correct2.append(count2)
            print("fear")
        else:
            error2.append(count2)
    elif ("Outcome: surprise" in result):
        if (sample.label == 5):
            correct2.append(count2)
            print("surprise")
        else:
            error2.append(count2)
    else: # error
        error2.append(count2)
        
    print(seconds, "seconds")

In [15]:
from tqdm import tqdm

eliminated = 0
for row in tqdm(new_df2.itertuples(index=False), total=new_df2.shape[0]):
    if count2 == new_resamples[eliminated]:
        #print(count)
        eliminated+=1
        run_fhe_bert_tiny2(row)
    count2+=1
    
    if eliminated == len(new_resamples):
        break

  0%|          | 0/60 [00:00<?, ?it/s]

sentence:  i feel so embarrassed


  2%|▏         | 1/60 [02:30<2:28:23, 150.91s/it]

[-2.1479444820452404e+48, -1.0245268924867246e+49, -2.505572917661846e+49, -1.947839894253201e+49, -3.0223249225881133e+49, -6.878018426431168e+49]
sadness
137.0 seconds
sentence:  i do feel stressed


  3%|▎         | 2/60 [05:01<2:25:28, 150.48s/it]

...

[5.344452449042286e+49, 6.219518247937191e+49, 1.5725584682300484e+49, 9.50311031998882e+49, -3.540281012432525e+48, 3.513942512789464e+49]
186.0 seconds
sentence:  i am feeling amazing and seeing the difference


 98%|█████████▊| 59/60 [2:19:51<03:06, 186.79s/it]

[-6.054457551256267e+49, -1.1478190133246162e+48, 1.0038729708527271e+49, -7.177732979431191e+49, 4.044109767044214e+49, -2.229629427524749e+49]
186.0 seconds
sentence:  i always feel very shocked by that me threatening


 98%|█████████▊| 59/60 [2:23:20<02:25, 145.77s/it]

[-2.7193861132967857e+48, 2.2035237037912235e+49, -5.74532076404295e+49, 1.6302376902285338e+49, -1.0489457100410133e+50, -7.749015641279432e+48]
196.0 seconds


In [16]:
len(correct2)

7

In [17]:
# total accuracy
total_accuracy = correct + correct2
total_accuracy.sort()
print(len(total_accuracy))

16


In [18]:
len(total_accuracy)/60

0.26666666666666666

In [19]:
# Each label accuracy
label0_accuracy = []
label1_accuracy = []
label2_accuracy = []
label3_accuracy = []
label4_accuracy = []
label5_accuracy = []

for i in total_accuracy:
    if i < 10:
        label0_accuracy.append(i)
    elif i < 20:
        label1_accuracy.append(i)
    elif i < 30:
        label2_accuracy.append(i)
    elif i < 40:
        label3_accuracy.append(i)
    elif i < 50:
        label4_accuracy.append(i)
    elif i < 60:
        label5_accuracy.append(i)

In [20]:
print(len(label0_accuracy))
print(len(label1_accuracy))
print(len(label2_accuracy))
print(len(label3_accuracy))
print(len(label4_accuracy))
print(len(label5_accuracy))

4
1
3
5
1
2


In [21]:
print(df_label[df_label['label'] == 0].shape[0])
print(df_label[df_label['label'] == 1].shape[0])
print(df_label[df_label['label'] == 2].shape[0])
print(df_label[df_label['label'] == 3].shape[0])
print(df_label[df_label['label'] == 4].shape[0])
print(df_label[df_label['label'] == 5].shape[0])


581
695
159
275
224
66


In [22]:
new_resamples

[0,
 1,
 2,
 3,
 4,
 5,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 26,
 27,
 28,
 30,
 31,
 32,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 44,
 45,
 46,
 47,
 48,
 49,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59]

In [23]:
# check final wrong answer
final_wrong = []

for i in range(60):
    if not i in total_accuracy and i in new_resamples:
        print(i)
        print(f"first: {output[i]}")
        print(f"retest: {output2[new_resamples.index(i)]}")
        final_wrong.append(i)

2
first: [-1.16584e+48, -5.1292e+49, 3.81851e+49, -3.54781e+49, 4.13336e+49, -3.14962e+48]
retest: [-1.3326942732166602e+48, -1.8974957360945132e+49, 5.052384800709916e+49, 6.064004695924198e+49, 1.802438456373948e+49, 1.9934872960647494e+49]
3
first: [7.90639e+48, 4.62542e+49, -8.74293e+48, -2.92132e+49, 3.07521e+49, 3.18034e+49]
retest: [3.93517028339959e+48, 7.9706638670716e+49, -4.223800872907661e+48, 6.356745147975318e+48, -6.134652498245184e+48, -8.2272726092797e+48]
4
first: [1.94879e+49, 2.86678e+49, -4.62297e+49, -3.37907e+49, -1.81598e+49, 2.46072e+49]
retest: [5.2588673014077754e+48, 4.1559264769768475e+49, -2.0500326623082804e+49, -1.472544756610881e+48, -1.1002328021071624e+48, 2.7034519433369745e+49]
5
first: [1.39257e+49, -7.47774e+49, 2.76157e+49, -4.10241e+49, 3.2083e+49, -2.511e+49]
retest: [-5.708559583110491e+48, 5.082878975933411e+49, 6.6736680394893515e+47, 5.217724573253766e+48, -7.749678282416502e+49, 5.719390876947541e+49]
8
first: [5.37873e+49, 4.11065e+49, 3.

In [24]:
print(len(final_wrong))

44


In [25]:
import numpy as np
np.savetxt("output2_labels4_60", output, delimiter=",")
np.savetxt("execution_time2_labels4_60", output, delimiter=",")